In [4]:
import tensorflow as tf
import numpy as np
import math

2025-08-26 13:16:30.098034: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-26 13:16:30.127340: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-26 13:16:30.226566: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756232190.408016   12225 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756232190.463267   12225 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756232190.597664   12225 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [5]:
def _soft_quantize(x, k, levels, thresholds):
    x_reshaped = tf.expand_dims(x, axis=-1)
    
    sigs = tf.sigmoid(k * (x_reshaped - thresholds))
    padded_sigs = tf.pad(sigs, [[0, 0]] * (len(sigs.shape)-1) + [[1, 1]], 
                            constant_values=0)
    padded_sigs = tf.tensor_scatter_nd_update(
        padded_sigs, 
        [[i, padded_sigs.shape[-1]-1] for i in range(tf.shape(padded_sigs)[0])],
        tf.ones(tf.shape(padded_sigs)[0])
    )
    # w_i = P(x > t_{i-1}) - P(x > t_i)
    weights = tf.abs(padded_sigs[..., 1:] - padded_sigs[..., :-1])  ##Some bug is ther ein the leves
    return tf.reduce_sum(weights * levels, axis=-1)

In [6]:
n_bits=2
initial_range=[-1.0, 1.0]
num_levels = 2**n_bits   

initial_levels = tf.linspace(initial_range[0], 
                        initial_range[1], 
                        num_levels)
        


# --- (X-axis) ---
initial_thresholds =  tf.linspace(initial_range[0], 
                                initial_range[1], 
                                num_levels - 1)

E0000 00:00:1756232210.474788   12225 cuda_executor.cc:1228] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1756232210.481545   12225 gpu_device.cc:2341] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [7]:
initial_levels

<tf.Tensor: shape=(4,), dtype=float32, numpy=array([-1.        , -0.3333333 ,  0.33333337,  1.        ], dtype=float32)>